# Física BR · Taller: Fuentes del Campo Magnético
### Física BR — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* Toda corriente produce un campo magnético. Este taller calcula
el campo de las geometrías más usadas en ingeniería —un alambre recto, una espira, un par de
conductores de ida y retorno y una espira cuadrada— a partir de la ley de Biot-Savart. La fuerza que
ese campo ejerce sobre cargas, conductores y bobinas está en el taller **Fuerza y Torque Magnéticos**;
el solenoide y el toroide, en el taller **Ley de Ampère**.

**Convenciones:** tres cifras significativas · coma decimal · µ₀ = 4π×10⁻⁷ T·m/A.

**Ecuaciones clave:**
- Biot-Savart:  dB = (µ₀/4π)·I·dL × r̂ / r²
- Alambre recto largo:  B = µ₀·I/(2π·r)
- Centro de una espira circular:  B = µ₀·I/(2R)
- Segmento recto finito:  B = µ₀·I/(4π·d)·(sen α₁ + sen α₂)

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*

---
## Punto 1 — Campo de un alambre recto (Ing. Eléctrica/Instalaciones)  ★★★  [PRC, ULC]
Un alambre recto largo conduce una corriente  I = 15,0 A. Se busca el campo magnético que produce a
una distancia  r = 0,0500 m  del alambre.

**Campo de alambre recto:**  B = μ₀·I/(2π·r)

> 🔧 **Aplicación en ingeniería:** toda línea con corriente genera un campo magnético a su alrededor.
> Esto importa para la interferencia electromagnética (EMI) en instalaciones, el diseño de blindajes
> y el cálculo de distancias entre conductores para evitar acoplamientos indeseados.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que el campo magnético de un alambre recto NO depende de la distancia, que es igual de fuerte cerca y lejos. Dame el mejor contraargumento."* Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

mu0 = 4*np.pi*1e-7
e = 1.60e-19

def esquema():
    fig,ax=plt.subplots(figsize=(6.5,5))
    # alambre (vertical, corriente hacia arriba)
    ax.plot([2,2],[0,4],color='#c0392b',lw=4,zorder=3)
    ax.annotate('',xy=(2,4.3),xytext=(2,3.8),arrowprops=dict(arrowstyle='-|>',color='#c0392b',lw=3))
    ax.text(2.2,4.2,'I',color='#c0392b',fontsize=12,fontweight='bold')
    # círculos de campo (concéntricos, más juntos cerca del alambre)
    from matplotlib.patches import Ellipse
    for r in [0.6,1.1,1.7]:
        ax.add_patch(Ellipse((2,2),r*2,r*0.7,fill=False,ec='#2980b9',lw=1.5,alpha=0.7))
    # flechas mostrando el sentido (regla mano derecha)
    for r in [0.6,1.1,1.7]:
        ax.annotate('',xy=(2+r,2.05),xytext=(2+r,1.95),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=1.5))
        ax.annotate('',xy=(2-r,1.95),xytext=(2-r,2.05),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=1.5))
    ax.text(4,2,'B',color='#2980b9',fontsize=12,fontweight='bold')
    ax.text(2,-0.5,'El campo forma CÍRCULOS alrededor del alambre: B = μ₀I/(2πr)\n(cae como 1/r; la mano derecha da el sentido)',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(-0.5,4.5); ax.set_ylim(-1,4.8); ax.axis('off')
    ax.set_title('Campo magnético de un alambre recto')
    plt.tight_layout(); plt.show()
esquema()

def campo_alambre(I=15.0, r_cm=5.0):
    r=r_cm/100; B=mu0*I/(2*np.pi*r)
    rs=np.linspace(0.01,0.2,100); Bs=mu0*I/(2*np.pi*rs)
    plt.figure(figsize=(9,4)); plt.plot(rs*100,Bs*1e6,lw=2,color='navy')
    plt.scatter([r_cm],[B*1e6],color='red',s=60,zorder=5,label=f'r={r_cm:.1f} cm → B={B*1e6:.1f} µT')
    plt.xlabel('distancia r (cm)'); plt.ylabel('campo B (µT)')
    plt.title('Campo de alambre recto (B ∝ 1/r)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(campo_alambre,
    I=FloatSlider(value=15.0,min=1,max=30,step=1,description='I (A)'),
    r_cm=FloatSlider(value=5.0,min=1,max=20,step=1,description='r (cm)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    I=15.0; r=0.05; B=mu0*I/(2*np.pi*r)
    _pc(f"Campo: B = μ0·I/(2π·r) = (4π×10⁻⁷ × 15,0)/(2π × 0,0500)")
    _pc(f"     = {B:.3g} T = {B*1e6:.3g} µT")
    _pc("El campo forma círculos alrededor del alambre y decae como 1/r.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Un alambre recto conduce 15,0 A. Encuentre el campo magnético a 0,0500 m de distancia (en µT).",60,0.05,"µT","B=μ0I/(2πr)=60,0 µT.", id_preg="Fuentes_p1_5", peso=3.0)
quiz_opcion_multiple("A medida que nos alejamos de un alambre recto con corriente, el campo magnético:",
    ["Aumenta","Disminuye como 1/r","No cambia","Disminuye como 1/r²"],1,
    "B=μ0I/(2πr): el campo decae proporcionalmente a 1/r.", id_preg="Fuentes_p1_6", peso=3.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante afirma que el campo de un alambre recto largo decae con 1/r², igual que el campo de una carga puntual. ¿Correcto o solo convincente?',
    ['Solo convincente: el alambre es una fuente lineal; al sumar sus elementos resulta B = μ₀I/(2πr), que decae con 1/r',
     'Correcto: cada elemento de corriente aporta según Biot-Savart con 1/r², así que el total también decae con 1/r²',
     'Correcto: todo campo producido por una fuente localizada decae con 1/r², sin importar su geometría',
     'Solo convincente: el campo no depende de r, porque la corriente es la misma a lo largo de todo el alambre'],
    0,
    'Cada elemento sí aporta con 1/r², pero al integrar sobre un alambre infinito el resultado es B = μ₀I/(2πr): 1/r. La opción de Biot-Savart es la más convincente y es falsa: confunde la ley de cada elemento con la del total.',
    id_preg='Fuentes_p1_concep_1', peso=2.0)


---
## Punto 2 — Campo en el centro de una espira (Ing. Eléctrica/Electrónica)  ★★  [CL, UCM]
Una espira circular de radio  R = 0,100 m  conduce una corriente  I = 8,00 A. Se busca el campo
magnético en su centro.

**Campo en el centro de una espira:**  B = μ₀·I/(2R)

> 🔧 **Aplicación en ingeniería:** las espiras y bobinas concentran el campo magnético en su centro.
> Son la base de electroimanes, sensores inductivos, bobinas de Helmholtz (que crean campos
> uniformes) y de la generación de campos controlados en instrumentación.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele: *"Créame una variante con otra espira: distinto radio y corriente, y pídeme calcular el campo en su centro con B=μ₀I/2R. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

mu0 = 4*np.pi*1e-7
e = 1.60e-19

def esquema():
    fig,ax=plt.subplots(figsize=(6.5,4.5))
    from matplotlib.patches import Ellipse
    # espira (elipse en perspectiva)
    ax.add_patch(Ellipse((2,2),3,1.2,fill=False,ec='#c0392b',lw=3))
    # corriente en la espira
    ax.annotate('',xy=(3.5,2.1),xytext=(3.5,1.9),arrowprops=dict(arrowstyle='-|>',color='#c0392b',lw=2))
    ax.text(3.7,2.2,'I',color='#c0392b',fontsize=11,fontweight='bold')
    # campo en el centro (sale perpendicular al plano de la espira)
    ax.annotate('',xy=(2,3.5),xytext=(2,2),arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=3))
    ax.text(2.2,3,'B',color='#2980b9',fontsize=12,fontweight='bold')
    ax.scatter([2],[2],s=60,color='#2c3e50',zorder=5)
    ax.text(2,0.6,'La corriente circular genera un campo en el CENTRO: B = μ₀I/(2R)\n(perpendicular al plano de la espira)',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(-0.5,4.5); ax.set_ylim(0,4); ax.axis('off')
    ax.set_title('Campo en el centro de una espira')
    plt.tight_layout(); plt.show()
esquema()

def campo_espira(I=8.0, R_cm=10.0):
    R=R_cm/100; B=mu0*I/(2*R)
    Rs=np.linspace(0.02,0.3,100); Bs=mu0*I/(2*Rs)
    plt.figure(figsize=(9,4)); plt.plot(Rs*100,Bs*1e6,lw=2,color='teal')
    plt.scatter([R_cm],[B*1e6],color='red',s=60,zorder=5,label=f'R={R_cm:.0f} cm → B={B*1e6:.1f} µT')
    plt.xlabel('radio de la espira R (cm)'); plt.ylabel('campo B (µT)')
    plt.title('Campo en el centro (menor R ⟹ mayor B)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(campo_espira,
    I=FloatSlider(value=8.0,min=1,max=20,step=1,description='I (A)'),
    R_cm=FloatSlider(value=10.0,min=2,max=30,step=2,description='R (cm)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    I=8.0; R=0.10; B=mu0*I/(2*R)
    _pc(f"Campo: B = μ0·I/(2R) = (4π×10⁻⁷ × 8,00)/(2 × 0,100)")
    _pc(f"     = {B:.3g} T = {B*1e6:.3g} µT")
    _pc("Una espira más pequeña (menor R) concentra más el campo en el centro.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Una espira de radio 0,100 m conduce 8,00 A. Encuentre el campo magnético en su centro (en µT).",50.3,0.05,"µT","B=μ0I/(2R)≈50,3 µT.", id_preg="Fuentes_p2_7", peso=2.0)
quiz_opcion_multiple("Para lograr un campo más intenso en el centro de una espira, con la misma corriente conviene:",
    ['Aumentar el radio (más alambre aporta)', 'Reducir el radio (B = μ₀I/2R)', 'Alejar el punto (el campo se concentra)', 'No se puede sin más corriente'],1,
    "B=μ0I/(2R): menor radio concentra más el campo.", id_preg="Fuentes_p2_8", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante afirma: 'si duplico el radio de la espira con la misma corriente, el campo en el centro se reduce a la mitad, aunque haya el doble de alambre'. ¿Correcto o solo convincente?",
    ['Correcto: hay el doble de alambre, pero cada tramo está al doble de distancia y aporta 1/4; el neto es 1/2',
     'Correcto: se reduce a la mitad porque la misma corriente se reparte entre el doble de alambre',
     'Solo convincente: el doble de alambre compensa exactamente la mayor distancia y el campo no cambia',
     'Solo convincente: con el doble de alambre aportando, el campo en el centro se duplica'],
    0,
    "El estudiante TIENE razón: B = μ₀I/(2R). Cada elemento aporta ∝ dl/R², y la longitud total es 2πR: (2πR)/R² ∝ 1/R. La opción «se reduce a la mitad porque…» acierta el veredicto con una razón falsa: la corriente es la misma en todo el alambre, no 'se reparte'.",
    id_preg='Fuentes_p2_concep_2', peso=2.0)


---
## Punto 3 — Ida y retorno juntos: la línea bifilar (Ing. Eléctrica/Compatibilidad electromagnética)  ★★  [PRC, ULC]
Un circuito lleva  **I = 20,0 A**  por un conductor y la devuelve por otro, paralelo y separado
**s = 1,00 cm**. Se busca el campo magnético en un punto P sobre la línea que une los conductores, a
**r = 10,0 cm**  del centro del par, y compararlo con el de un solo alambre.

**Campo de cada conductor:**  B = µ₀·I/(2π·d)    **Lejos del par (r ≫ s):**  B ≈ µ₀·I·s/(2π·r²)

> 🔧 **Aplicación en ingeniería:** por eso en instalaciones eléctricas los conductores de ida y de
> retorno se tienden juntos, y en cables de señal se trenzan: sus campos casi se cancelan lejos del
> cable. Así se reducen la interferencia sobre equipos sensibles y los campos de fuga en tableros,
> bandejas portacables y cuartos eléctricos.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Calculé el campo de dos conductores con corrientes opuestas en un punto a 10 cm. Actúa como evaluador escéptico: pregúntame qué sentido tiene el campo de cada conductor en P, por qué los resté y qué pasa con el resultado si me alejo diez veces más. No me des la respuesta."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

from ipywidgets import IntSlider, Checkbox

def _fmt(x, cs=3):
    # 3 cifras significativas, notación ×10ⁿ si hace falta, coma decimal
    if x == 0: return "0"
    _sup = '⁰¹²³⁴⁵⁶⁷⁸⁹'
    exp = int(np.floor(np.log10(abs(x))))
    if -2 <= exp <= 3:
        dec = max(cs-1-exp, 0)
        t = f"{x:.{dec}f}"
    else:
        mant = x/10**exp
        if abs(round(mant, cs-1)) >= 10: mant, exp = mant/10, exp+1
        e = ('⁻' if exp < 0 else '') + ''.join(_sup[int(d)] for d in str(abs(exp)))
        t = f"{mant:.{cs-1}f}×10{e}"
    return _num_coma(t)

mu0 = 4*np.pi*1e-7
I3, s3, r3 = 20.0, 0.0100, 0.100     # A, separación (m), distancia de P al centro (m)

def _B_bif(x, I=I3, s=s3):
    # campo (componente y) sobre el eje x de dos alambres en x = ∓s/2 con corrientes +z y −z
    return mu0*I/(2*np.pi)*(1/(x + s/2) - 1/(x - s/2))

def esquema():
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.axhline(0, color='#bdc3c7', lw=1)
    ax.scatter([-0.6], [0], s=420, facecolor='white', edgecolor='black', zorder=5); ax.scatter([-0.6], [0], s=40, color='black', zorder=6)
    ax.scatter([0.6], [0], s=420, facecolor='white', edgecolor='black', zorder=5); ax.text(0.6, 0, '×', ha='center', va='center', fontsize=15, zorder=6)
    ax.text(-0.6, -0.55, 'ida: +I', ha='center', fontsize=9); ax.text(0.6, -0.55, 'retorno: −I', ha='center', fontsize=9)
    ax.annotate('', xy=(0.6, 0.45), xytext=(-0.6, 0.45), arrowprops=dict(arrowstyle='<->', color='#7f8c8d'))
    ax.text(0, 0.6, 's = 1,00 cm', ha='center', fontsize=9, color='#7f8c8d')
    ax.scatter([6.0], [0], s=60, color='#c0392b', zorder=5); ax.text(6.0, 0.3, 'P', color='#c0392b', fontsize=12, fontweight='bold', ha='center')
    ax.annotate('', xy=(6.0, -0.75), xytext=(0, -0.75), arrowprops=dict(arrowstyle='<->', color='#2c3e50'))
    ax.text(3.0, -1.05, 'r = 10,0 cm desde el centro del par', ha='center', fontsize=9)
    ax.annotate('', xy=(6.0, 1.3), xytext=(6.0, 0.05), arrowprops=dict(arrowstyle='-|>', color='#2980b9', lw=2))
    ax.annotate('', xy=(6.25, -1.45), xytext=(6.25, 0.05), arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=2))
    ax.text(5.15, 1.15, 'B de la ida', color='#2980b9', fontsize=8.5)
    ax.text(6.35, -1.3, 'B del retorno\n(más cercano: un poco mayor)', color='#e67e22', fontsize=8.5)
    ax.set_xlim(-1.5, 9.0); ax.set_ylim(-1.7, 1.6); ax.axis('off')
    ax.set_title('Línea bifilar: dos conductores con corrientes opuestas (vista de frente)')
    plt.tight_layout(); plt.show()
esquema()

def explorar_bifilar(I=20.0, s_cm=1.00, r_max_cm=100.0):
    s = s_cm/100
    r = np.linspace(max(2*s, 0.02), r_max_cm/100, 300)
    Bb = np.abs(_B_bif(r, I, s)); B1 = mu0*I/(2*np.pi*r)
    fig, ax = plt.subplots(figsize=(9, 3.9))
    ax.loglog(r*100, B1*1e6, color='#2980b9', lw=2, label='un solo alambre (∝ 1/r)')
    ax.loglog(r*100, Bb*1e6, color='#c0392b', lw=2.5, label='ida y retorno juntos (∝ 1/r²)')
    ax.set_xlabel('distancia r (cm)'); ax.set_ylabel('B (µT)')
    r1 = 1.00
    ax.set_title(f'A 1,00 m: un alambre = {_fmt(mu0*I/(2*np.pi*r1)*1e6)} µT · par bifilar = {_fmt(abs(_B_bif(r1, I, s))*1e6)} µT')
    ax.legend(); ax.grid(alpha=.3, which='both'); plt.tight_layout(); plt.show()

interact(explorar_bifilar,
    I=FloatSlider(value=20.0, min=5.0, max=50.0, step=5.0, description='I (A)'),
    s_cm=FloatSlider(value=1.00, min=0.50, max=5.00, step=0.50, readout_format='.2f', description='s (cm)'),
    r_max_cm=FloatSlider(value=100.0, min=20.0, max=300.0, step=20.0, description='r máx (cm)'));

def solucion():
    d1, d2 = r3 + s3/2, r3 - s3/2
    B1 = mu0*I3/(2*np.pi*d1); B2 = mu0*I3/(2*np.pi*d2)
    Bn = B2 - B1
    print_es("Cada alambre produce B = µ₀·I/(2π·d), tangente a círculos centrados en él (regla de la mano derecha).")
    print_es("En P los dos campos son paralelos y de sentidos opuestos: se restan.")
    print_es("")
    print_es(f"   Ida (d₁ = 10,5 cm):     B₁ = 2,00×10⁻⁷·20,0/0,105 = {_fmt(B1*1e6)} µT")
    print_es(f"   Retorno (d₂ = 9,50 cm): B₂ = 2,00×10⁻⁷·20,0/0,0950 = {_fmt(B2*1e6)} µT")
    print_es(f"   Campo neto: B = B₂ − B₁ = {_fmt(Bn*1e6)} µT  (domina el conductor más cercano)")
    print_es("")
    print_es(f"   Comparación: un solo alambre a 10,0 cm daría {_fmt(mu0*I3/(2*np.pi*r3)*1e6)} µT, diez veces más.")
    print_es("   Lejos del par (r ≫ s):  B ≈ µ₀·I·s/(2π·r²)")
    print_es(f"   Verificación: 2,00×10⁻⁷·20,0·0,0100/0,100² = {_fmt(mu0*I3*s3/(2*np.pi*r3**2)*1e6)} µT ✓")
    print_es("   Al alejarse, el campo del par cae como 1/r²: a 1,00 m es 100 veces menor que el de un alambre.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Encuentre el campo magnético neto en P, a 10,0 cm del centro del par bifilar (en µT).",
    4.01, 0.03, "µT", "B = B₂ − B₁ = 42,1 − 38,1 = 4,01 µT (o con la aproximación µ₀·I·s/(2π·r²) = 4,00 µT).", id_preg="Fuentes_p3_1", peso=2.0)
quiz_numerico("¿Qué campo produciría en el mismo punto un solo alambre con 20,0 A, sin el conductor de retorno? (en µT)",
    40.0, 0.03, "µT", "B = µ₀·I/(2π·r) = 2,00×10⁻⁷·20,0/0,100 = 40,0 µT: diez veces el campo del par.", id_preg="Fuentes_p3_2", peso=2.0)
quiz_opcion_multiple("A 1,00 m del par, ¿cuántas veces menor es el campo de la línea bifilar que el de un solo alambre con la misma corriente?",
    ["Unas 10 veces, igual que a 10,0 cm, porque la proporción no cambia", "Unas 100 veces: el par cae como 1/r² y el alambre solo como 1/r", "Unas 2 veces, porque hay dos conductores que se reparten el campo", "Infinitas veces: a 1,00 m el campo del par ya es exactamente cero"], 1,
    "Razón = [µ₀Is/(2πr²)]/[µ₀I/(2πr)] = s/r: a 10,0 cm es 1/10 y a 1,00 m es 1/100. El par no se anula: se debilita más rápido.",
    id_preg="Fuentes_p3_3", peso=2.0)

quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante afirma que el campo de la línea bifilar es CERO en todo punto exterior, porque las dos corrientes son iguales y opuestas. ¿Correcto o solo convincente?',
    ['Solo convincente: el conductor más cercano domina; en P quedan 4,01 µT y el campo cae como 1/r²',
     'Correcto: dos corrientes iguales y opuestas siempre producen campos que se cancelan exactamente',
     'Correcto: por la ley de Ampère, una trayectoria que rodee ambos conductores encierra corriente cero',
     'Solo convincente: el campo es el doble del de un alambre, porque hay dos conductores'],
    0,
    'Los conductores no están en el mismo lugar: en P uno está más cerca y su campo es mayor. La opción de Ampère es la más tentadora: corriente encerrada cero da ∮B·dl = 0, no B = 0 en cada punto, porque aquí el campo no tiene la simetría que permitiría sacarlo de la integral.',
    id_preg='Fuentes_p3_concep_3', peso=2.0)


---
## Punto 4 — Espira cuadrada: Biot-Savart por segmentos (Ing. Electrónica/Sensores)  ★★★  [PRC, UCM]
Una espira cuadrada de lado  **a = 20,0 cm**  conduce  **I = 5,00 A**. Cada lado es un segmento recto
finito; su campo en el centro se obtiene integrando la ley de Biot-Savart. Se busca el campo en el
centro y se compara con el de una espira **circular hecha con el mismo alambre** (0,800 m).

**Segmento finito:**  B = µ₀·I/(4π·d)·(sen α₁ + sen α₂)  con  d = a/2  y  α₁ = α₂ = 45,0°
**Espira circular:**  B = µ₀·I/(2R)

> 🔧 **Aplicación en ingeniería:** los detectores de vehículos en semáforos y peajes son espiras
> rectangulares enterradas en el pavimento: el campo que producen, y cómo cambia cuando un vehículo
> pasa encima, depende de su geometría. Elegir la forma de una bobina es una decisión de diseño.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega la integral de Biot-Savart para un segmento y pídele: *"Tradúceme esta integral a un dibujo en palabras: qué representan d, α₁ y α₂, y por qué cada lado de la espira aporta en el mismo sentido en el centro. Luego pregúntame cómo cambiaría el resultado con un hexágono. No me des la respuesta."* Inténtalo tú primero.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---

from ipywidgets import IntSlider, Checkbox

def _fmt(x, cs=3):
    # 3 cifras significativas, notación ×10ⁿ si hace falta, coma decimal
    if x == 0: return "0"
    _sup = '⁰¹²³⁴⁵⁶⁷⁸⁹'
    exp = int(np.floor(np.log10(abs(x))))
    if -2 <= exp <= 3:
        dec = max(cs-1-exp, 0)
        t = f"{x:.{dec}f}"
    else:
        mant = x/10**exp
        if abs(round(mant, cs-1)) >= 10: mant, exp = mant/10, exp+1
        e = ('⁻' if exp < 0 else '') + ''.join(_sup[int(d)] for d in str(abs(exp)))
        t = f"{mant:.{cs-1}f}×10{e}"
    return _num_coma(t)

mu0 = 4*np.pi*1e-7
I4, a4 = 5.00, 0.200      # A, lado de la espira cuadrada (m)

def _B_lado(I, d, al1, al2):
    # segmento recto finito (Biot-Savart): B = µ0·I/(4π·d)·(sen α1 + sen α2)
    return mu0*I/(4*np.pi*d)*(np.sin(al1) + np.sin(al2))

def esquema():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
    from matplotlib.patches import Rectangle, Circle
    ax1.add_patch(Rectangle((-1, -1), 2, 2, fill=False, ec='#c0392b', lw=3))
    for (x, y, dx, dy) in [(0, -1, 0.3, 0), (1, 0, 0, 0.3), (0, 1, -0.3, 0), (-1, 0, 0, -0.3)]:
        ax1.annotate('', xy=(x+dx, y+dy), xytext=(x, y), arrowprops=dict(arrowstyle='-|>', color='#c0392b', lw=2))
    ax1.scatter([0], [0], s=40, color='black'); ax1.text(0.08, 0.08, 'centro', fontsize=8)
    ax1.plot([0, 0], [0, -1], color='#7f8c8d', ls='--'); ax1.text(0.05, -0.55, 'd = a/2', fontsize=8.5, color='#7f8c8d')
    ax1.plot([0, 1], [0, -1], color='#7f8c8d', ls=':'); ax1.plot([0, -1], [0, -1], color='#7f8c8d', ls=':')
    ax1.text(0.18, -0.35, 'α = 45°', fontsize=8.5, color='#7f8c8d')
    ax1.text(0, -1.3, 'a = 20,0 cm · I = 5,00 A', ha='center', fontsize=9)
    ax1.set_xlim(-1.4, 1.4); ax1.set_ylim(-1.5, 1.3); ax1.set_aspect('equal'); ax1.axis('off')
    ax1.set_title('Espira cuadrada (4 segmentos finitos)')
    R = 4*1/(2*np.pi)
    ax2.add_patch(Circle((0, 0), R, fill=False, ec='#2980b9', lw=3)); ax2.scatter([0], [0], s=40, color='black')
    ax2.plot([0, R], [0, 0], color='#7f8c8d', ls='--'); ax2.text(0.15, 0.06, 'R = 12,7 cm', fontsize=8.5, color='#7f8c8d')
    ax2.text(0, -1.3, 'misma longitud de alambre: 0,800 m', ha='center', fontsize=9)
    ax2.set_xlim(-1.4, 1.4); ax2.set_ylim(-1.5, 1.3); ax2.set_aspect('equal'); ax2.axis('off')
    ax2.set_title('Espira circular del mismo perímetro')
    plt.tight_layout(); plt.show()
esquema()

def explorar_poligono(lados=4, I=5.00, perimetro_m=0.800):
    n = int(lados)
    a = perimetro_m/n                 # lado del polígono regular
    d = a/(2*np.tan(np.pi/n))         # apotema: distancia del centro a cada lado
    al = np.pi/n                      # medio ángulo subtendido por cada lado
    Bn = n*_B_lado(I, d, al, al)
    Bc = mu0*I/(2*(perimetro_m/(2*np.pi)))
    ns = np.arange(3, 13)
    Bs = [k*_B_lado(I, (perimetro_m/k)/(2*np.tan(np.pi/k)), np.pi/k, np.pi/k) for k in ns]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(ns, np.array(Bs)*1e6, 'o-', color='#c0392b', label='polígono regular')
    ax.axhline(Bc*1e6, color='#2980b9', ls='--', label=f'círculo: {_fmt(Bc*1e6)} µT')
    ax.scatter([n], [Bn*1e6], s=120, color='black', zorder=5)
    ax.set_xlabel('número de lados'); ax.set_ylabel('B en el centro (µT)')
    ax.set_title(f'{n} lados: B = {_fmt(Bn*1e6)} µT · {_fmt(Bn/Bc*100)} % del círculo de igual perímetro')
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

interact(explorar_poligono,
    lados=IntSlider(value=4, min=3, max=12, step=1, description='lados'),
    I=FloatSlider(value=5.00, min=1.00, max=10.0, step=0.50, readout_format='.2f', description='I (A)'),
    perimetro_m=FloatSlider(value=0.800, min=0.200, max=2.00, step=0.100, readout_format='.3f', description='perímetro (m)'));

def solucion():
    d = a4/2
    Bl = _B_lado(I4, d, np.pi/4, np.pi/4)
    Bt = 4*Bl
    R = 4*a4/(2*np.pi); Bc = mu0*I4/(2*R)
    print_es("Biot-Savart para un segmento recto finito, visto desde un punto a distancia d:")
    print_es("   B = µ₀·I/(4π·d)·(sen α₁ + sen α₂)")
    print_es("")
    print_es("Un lado (d = a/2 = 0,100 m; α₁ = α₂ = 45,0°):")
    print_es(f"   B_lado = 1,00×10⁻⁷·5,00/0,100·(2·sen 45,0°) = {_fmt(Bl*1e6)} µT")
    print_es("Los cuatro lados aportan en el mismo sentido (perpendicular a la espira):")
    print_es(f"   B = 4·B_lado = 2√2·µ₀·I/(π·a) = {_fmt(Bt*1e6)} µT")
    print_es("")
    print_es(f"Círculo con el mismo alambre (perímetro 0,800 m → R = 0,800/(2π) = {_fmt(R)} m):")
    print_es(f"   B = µ₀·I/(2R) = {_fmt(Bc*1e6)} µT")
    print_es(f"   La espira cuadrada da {_fmt(Bt/Bc)} veces el campo de la circular: {_fmt((Bt/Bc-1)*100)} % más.")
    print_es("   Razón: sus lados pasan a 10,0 cm del centro, más cerca que el círculo (12,7 cm).")
    print_es("   Verificación: con muchos lados el polígono tiende al círculo (pruébalo con el control).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Encuentre el campo que produce UN lado de la espira cuadrada en su centro (en µT).",
    7.07, 0.03, "µT", "B_lado = µ₀·I/(4π·d)·(2·sen 45,0°) = 1,00×10⁻⁷·5,00/0,100·1,414 = 7,07 µT.", id_preg="Fuentes_p4_1", peso=2.0)
quiz_numerico("Encuentre el campo total en el centro de la espira cuadrada (en µT).",
    28.3, 0.03, "µT", "B = 4·7,07 = 28,3 µT (todos los lados aportan en el mismo sentido).", id_preg="Fuentes_p4_2", peso=2.0)
quiz_numerico("Con el mismo alambre (0,800 m) se forma una espira circular. Encuentre el campo en su centro (en µT).",
    24.7, 0.03, "µT", "R = 0,800/(2π) = 0,127 m; B = µ₀·I/(2R) = 4π×10⁻⁷·5,00/0,255 = 24,7 µT.", id_preg="Fuentes_p4_3", peso=2.0)

quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: con el mismo alambre (0,800 m) y la misma corriente, un estudiante afirma que la espira circular produce más campo en su centro que la cuadrada, 'porque es más simétrica'. ¿Correcto o solo convincente?",
    ['Solo convincente: la cuadrada da 28,3 µT y la circular 24,7 µT; sus lados pasan más cerca',
     'Correcto: el círculo es la figura que concentra mejor el campo en su centro, por simetría',
     'Correcto: las esquinas del cuadrado cancelan parte del campo de los lados vecinos',
     'Solo convincente: las dos dan el mismo campo, porque el alambre y la corriente son iguales'],
    0,
    'La espira cuadrada da un 14,6 % más: el centro está a 10,0 cm de cada lado y a 12,7 cm del círculo. Simetría no significa más campo. Las esquinas no restan: todos los elementos aportan en el mismo sentido en el centro, solo que los de las esquinas aportan menos.',
    id_preg='Fuentes_p4_concep_4', peso=2.0)


---
## ✅ Fin del taller de Fuentes del Campo Magnético (Física BR)
Calculaste el campo de un alambre recto, de una espira, de un par de conductores de ida y retorno y de
una espira cuadrada. Todos salen de la misma ley de Biot-Savart: la geometría decide cómo se suman
las contribuciones. Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


# Calificación formativa del taller
calificacion_final(total_preguntas=14)
